## 1. 커스텀 Tool 설계 원칙

ReAct 에이전트에서 Tool은 에이전트가 외부 세계와 상호작용하는 유일한 통로이다. 따라서 Tool의 설계 품질이 에이전트의 전체 성능을 좌우한다. 다음은 커스텀 Tool을 설계할 때 반드시 지켜야 할 핵심 원칙들이다.

### 1.1 단일 책임 원칙 (Single Responsibility)

하나의 Tool은 **하나의 명확한 기능**만 수행해야 한다. 뉴스 검색과 요약을 하나의 Tool에 넣는 것이 아니라, 검색 Tool과 요약 Tool을 분리하여 구현한다. 이렇게 하면 LLM이 각 도구의 역할을 명확히 이해할 수 있고, 도구 조합의 유연성이 높아진다.

### 1.2 명확한 Tool 설명문 작성

Tool의 설명문(description)은 LLM이 도구를 올바르게 선택하는 데 결정적인 역할을 한다. 설명문에는 다음 요소가 포함되어야 한다:

- **기능 설명**: 이 도구가 무엇을 하는지 한 문장으로 명시
- **입력 형식**: 어떤 형식의 입력을 기대하는지 구체적으로 명시
- **사용 예시**: 실제 호출 예시를 포함하여 LLM이 패턴을 학습하도록 유도

### 1.3 입력/출력 형식 표준화

모든 Tool은 **문자열 입력 -> 문자열 출력** 형태를 유지하는 것이 좋다. ReAct 프레임워크에서 Action과 Observation은 텍스트 기반이므로, 일관된 문자열 인터페이스를 유지하면 에이전트와의 통합이 용이하다.

### 1.4 에러 처리와 Fallback 전략

외부 API를 호출하는 Tool은 반드시 예외 처리를 포함해야 한다. 네트워크 오류, 타임아웃, API 제한 등 다양한 실패 상황에 대비하여:

- try/except로 예외를 포착하고 의미 있는 오류 메시지를 반환
- 타임아웃 설정으로 무한 대기 방지
- 재시도 로직으로 일시적 오류에 대응
- 대체 데이터 소스나 기본값을 준비

## 2. 뉴스 검색 커스텀 Tool 개발

첫 번째 커스텀 Tool로 Google News RSS 피드를 활용한 **뉴스 검색 Tool**을 구현한다. 이 Tool은 실시간 뉴스 데이터를 수집하여 에이전트에게 최신 정보를 제공하는 역할을 한다.

### RSS 기반 뉴스 검색의 장점

- 별도의 API 키 없이 무료로 사용 가능
- XML 형식으로 구조화된 데이터 제공
- 실시간 최신 뉴스 접근 가능
- 한국어 뉴스 지원


- https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko

In [4]:
import os
import json
import requests
import xml.etree.ElementTree as ET
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def fetch_news(query, max_results=3):
    '''Goole News RSS에서 뉴스를 검색합니다.'''
    try:
        url = f'https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko'
        response = requests.get(url,timeout=10)        

        root = ET.fromstring(response.content)
        items = root.findall('.//item')[:max_results]  # .현재노드  // 모든 하위 태그탐색  item  <item>태그
        results = []
        for item in items:
            title = item.find('title').text if item.find('title') is not None else 'N/A'
            pub_date = item.find('pubDate').text if item.find('title') is not None else 'N/A'
            source = item.find('source').text if item.find('title') is not None else 'N/A'
            results.append({'title':title, 'date':pub_date, 'source':source})
        if results:
            output = f'{query}관련 최신 뉴스 {len(results)}건\n'
            for i, r in enumerate(results,1):
                output += f"  {i}. [{r['source']} {r['title']}  {r['date']}]"
            return output
        return f'{query}에 대한 뉴스를 찾을 수 없습니다'
    except Exception as e:
        return f'뉴스검색오류 : {e}'

print(fetch_news("지하철"))

지하철관련 최신 뉴스 3건
  1. [경향신문 [속보]지하철 홍대입구역~을지로입구역 운행 재개 - 경향신문  Thu, 28 May 2026 15:00:00 GMT]  2. [머니투데이 지하철서 잠든 여성만 노린 연쇄 성추행범, 영국서 '징역 4년' - 머니투데이 - 머니투데이  Thu, 28 May 2026 11:30:12 GMT]  3. [v.daum.net 지하철 2호선 전 구간 운행 재개…안전점검 완료 - v.daum.net  Thu, 28 May 2026 22:06:05 GMT]


In [3]:
query = '지하철'
url = f'https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko'
response = requests.get(url,timeout=10)        
response.content

b'<?xml version="1.0" encoding="UTF-8" standalone="yes"?><rss version="2.0" xmlns:media="http://search.yahoo.com/mrss/"><channel><generator>NFE/5.0</generator><title>"\xec\xa7\x80\xed\x95\x98\xec\xb2\xa0" - Google \xeb\x89\xb4\xec\x8a\xa4</title><link>https://news.google.com/search?q=%EC%A7%80%ED%95%98%EC%B2%A0&amp;hl=ko&amp;gl=KR&amp;ceid=KR:ko</link><language>ko</language><webMaster>news-webmaster@google.com</webMaster><copyright>Copyright \xc2\xa9 2026 Google. All rights reserved. This XML feed is made available solely for the purpose of rendering Google News results within a personal feed reader for personal, non-commercial use. Any other use of the feed is expressly prohibited. By accessing this feed or using these results in any manner whatsoever, you agree to be bound by the foregoing restrictions.</copyright><lastBuildDate>Fri, 29 May 2026 01:32:22 GMT</lastBuildDate><image><title>Google \xeb\x89\xb4\xec\x8a\xa4</title><url>https://lh3.googleusercontent.com/-DR60l-K8vnyi99NZovm